# 09. Statistical Evaluation: TRTR vs TSTR Benchmark

This notebook provides rigorous statistical comparison between:
- **Model A (TRTR)**: Train on Real data, Test on Real data (Baseline)
- **Model B (TSTR)**: Train on Synthetic data, Test on Real data (CVAE-augmented)

## Statistical Design

- **slide-Level Stratification**: Bootstrap samples slides, not spectra
- **Honest Predictions**: Each slide predicted ONLY by models from its validation fold
- **No Data Leakage**: Fold i models never predict on fold i's slides (they trained on them)
- **Cached Predictions**: Predictions computed once, reused across bootstrap iterations
- **Resampling**: Stratified bootstrap at slide level (cluster bootstrap)
- **Test**: Wilcoxon signed-rank test on bootstrap differences
- **Multiple testing**: Bonferroni correction across 4 models
- **Metrics**: Accuracy, F1-macro, Cohen's κ, McNemar test

## Section 1: Imports & Setup

In [1]:
import os
import sys
import json
import logging
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import tensorflow as tf
from datetime import datetime
from pathlib import Path
from scipy.stats import binomtest
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    cohen_kappa_score
)

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.utils.config_manager import ConfigManager
from src.notebooks.experiment_utils import setup_experiment
from src.data.loader import GliomaDataLoader
from src.models.cvae import CVAE
from src.utils.statistical_evaluation import (
    find_model_path,
    load_and_predict,
    get_fold_specific_predictions,
    stratified_slide_bootstrap,
    compute_cbr_predictions_per_fold,
    stratified_slide_bootstrap_cbr
)

print("✓ Imports successful")

c:\Users\Andrei\anaconda3\envs\glioma\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


✓ Imports successful


In [2]:
# Load configuration
config_name = 'exp_104'
config, output_dir = setup_experiment(config_name)
REGIMEN = "TSRTR"

CFG = {
    "data_path": config['paths']['processed_file'],
    "split_file": config['paths']['splits_file'],
    "model_a_dir": config['paths']['models_trtr_dir'],  # TRTR
    "model_b_dir": config['paths']['models_tstr_dir'],      # TSTR
    "output_dir": config['paths']['statistical_dir'],
    "n_bootstrap": config['statistics']['bootstrap_samples'],
    "alpha": config['statistics']['alpha'],
    "random_seed": config['statistics']['random_seed'],
    "models": {
        "Apollo": {"prefix": "apollo", "ext": ".joblib", "type": "sklearn"},
        "RF_Standard": {"prefix": "rf", "ext": ".joblib", "type": "sklearn"},
        "XGBoost": {"prefix": "xgb", "ext": ".json", "type": "xgboost"},
        "1D_CNN": {"prefix": "cnn", "ext": ".keras", "type": "keras"},
    },
    "primary_metric": config['statistics']['primary_metric'],
    "n_folds": config['cv_strategy']['n_folds']
}
if REGIMEN == "TSRTR":
    CFG["model_b_dir"] = config['paths']['models_tsrtr_dir']  # TSR TR

config["cv_strategy"]["slide_cap"] = None
np.random.seed(CFG["random_seed"])
os.makedirs(CFG["output_dir"], exist_ok=True)

# Add CVAE config for Classification by Reconstruction
CFG["cvae_dir"] = config['paths']['models_cvae_dir']

print(f"✓ Configuration loaded: {config_name}")
print(f"  Bootstrap samples: {CFG['n_bootstrap']}")
print(f"  α = {CFG['alpha']}")
print(f"  Number of folds: {CFG['n_folds']}")
print(f"  (CVAE config will be built after loading data to compute actual input shape)")



EXPERIMENT SETUP

Working directory: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data

 Loaded configuration: exp_104
   Description: TSAR Regimen: Fingerprint crop (700-1800), patient cap (500), and dynamic Raman augmentation

 Output directory: C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\results\results
  Experiment already registered (re-running)

--------------------------------------------------------------------------------
Configuration Summary:
--------------------------------------------------------------------------------
  Preprocessing:
    - AirPLS lambda: 100
    - Signal length: 1738
    - Crop region: [700, 1800]

  CV Strategy:
    - Split source: data/splits/strict_splits.json
    - Use group-aware CV: True
    - Hold out test set: True

  Models:
    - RF n_estimators: 100
    - SVM C: 1.0
    - CVAE epochs: 70


✓ Configuration loaded: exp_104
  Bootstrap samples: 1000
  α = 0.05
  Number of folds: 5
  (CVAE config will be built after loading dat

## Section 2: Load Data & Build slide-to-Fold Mapping

In [3]:
# Load data
print("Loading data...")
loader = GliomaDataLoader.from_config(
    h5_data_path=CFG['data_path'],
    config=config,
    memory_mapped=False
)

# Compute actual input shape from loaded data (handles preprocessing/cropping)
actual_n_features = loader.X.shape[1]
print(f"✓ Actual input shape: ({actual_n_features}, 1)")

# Build CVAE config with computed input shape
num_classes = config['models']['cnn']['num_classes']
CVAE_CONFIG = {
    'input_shape': (actual_n_features, 1),
    'num_classes': num_classes,
    'latent_dim': config['models']['cvae']['latent_dim'],
    'loss_weights': {
        'beta': config['models']['cvae']['loss_weights']['beta'],
        'alpha': config['models']['cvae']['loss_weights']['alpha']
    },
    'encoder': {
        'filters': config['models']['cvae']['encoder']['filters'],
        'kernels': config['models']['cvae']['encoder']['kernels'],
        'strides': config['models']['cvae']['encoder']['strides'],
        'use_global_pooling': False,
        'use_skip_gap': False
    },
    'decoder': {
        'initial_filters': config['models']['cvae']['decoder']['initial_filters'],
        'use_dense_projection': True,
        'filters': config['models']['cvae']['decoder']['filters'],
        'kernels': config['models']['cvae']['decoder']['kernels'],
        'strides': config['models']['cvae']['decoder']['strides']
    }
}

print(f"  CVAE latent dim: {CVAE_CONFIG['latent_dim']}")
print(f"  CVAE num classes: {CVAE_CONFIG['num_classes']}")

# ---------------------------------------------------------
# SECTION 2: Extract Pure Iterator Data (Perfectly Scaled)
# ---------------------------------------------------------
with open(CFG['split_file'], 'r') as f:
    splits_data = json.load(f)
splits = splits_data.get('cv_folds', splits_data)

print("\nExtracting preprocessed validation data directly from iterator...")
loader_iter = iter(loader)
all_groups = np.array(loader.groups)

X_val_list = []
y_val_list = []
slide_ids_list = []

for fold_idx in range(CFG['n_folds']):
    fold_data = next(loader_iter)
    fold_name = fold_data['fold_name']
    
    X_val = fold_data['val'][0].astype(np.float32)
    y_val = fold_data['val'][1].copy()
    
    # Replicate Notebook 02 label mapping to ensure matching classes
    if np.min(y_val) == 1:
        y_val = y_val - 1
    
    # Apply binary conversion if needed (for exp_105)
    if num_classes == 2:
        y_val = np.where(y_val < 3, 0, 1)

    # Get slide IDs using the string lookup
    raw_string_ids = splits[fold_name]['test']
    val_indices_sorted = np.where(np.isin(all_groups, raw_string_ids))[0]
    fold_sample_ids = all_groups[val_indices_sorted]
    fold_slide_ids = np.array([s.split('_')[0] + '_' + s.split('_')[1] for s in fold_sample_ids])
    
    X_val_list.append(X_val)
    y_val_list.append(y_val)
    slide_ids_list.append(fold_slide_ids)
    print(f"  ✓ {fold_name}: {len(y_val)} spectra mapped to {len(np.unique(fold_slide_ids))} slides")

# Combine into unified master arrays for the bootstrap
y_all = np.concatenate(y_val_list)
slide_ids_all = np.concatenate(slide_ids_list)
print(f"\n✓ Unified Test Set: {len(y_all)} spectra ready for evaluation.")
if num_classes == 2:
    print(f"  ✓ Binary classification task detected (Classes: {np.unique(y_all)})")


Loading data...
Loading CV splits from: data/splits/strict_splits.json
Loading data from results\exp_102\data\glioma_dataset.h5...
[*] 'tumor_only=True': Filtering out healthy tissue...
    Filtered to 226284 Pure Tumor Spectra.
[OK] Spectral Crop: [700, 1800] cm-1 -> indices [337, 908]
  Wavenumber mapping: 699.91 -> 1799.14 cm-1
  Renormalizing to cropped region...
    After renormalization: min=0.000000, max=1.000000 (fingerprint region)
  Spectra shape after crop: (226284, 571)
✓ Actual input shape: (571, 1)
  CVAE latent dim: 32
  CVAE num classes: 6

Extracting preprocessed validation data directly from iterator...

 Using CV splits from: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\data\splits\strict_splits.json
    Found 'cv_folds' section with 5 folds
   ! Separate held-out test set exists: 6 samples
   !  These samples will NOT be used in CV training/validation
   Fold 0: 160084 train samples, 45936 val samples
  ✓ fold_0: 45936 spectra mapped to 10 slides
   Fold 

In [4]:
cbr_preds, cbr_slide_ids, cbr_y_true = compute_cbr_predictions_per_fold(
    loader=loader, 
    cvae_config=CVAE_CONFIG, 
    model_dir=CFG['cvae_dir'], 
    n_folds=CFG['n_folds']
)


PHASE 1: Computing Classification by Reconstruction per-fold

 Using CV splits from: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\data\splits\strict_splits.json
    Found 'cv_folds' section with 5 folds
   ! Separate held-out test set exists: 6 samples
   !  These samples will NOT be used in CV training/validation
   Fold 0: 160084 train samples, 45936 val samples

fold_0...

  ✓ Accuracy: 0.1970
   Fold 1: 163393 train samples, 42627 val samples

fold_1...
  ✓ Accuracy: 0.2638
   Fold 2: 167257 train samples, 38763 val samples

fold_2...
  ✓ Accuracy: 0.2419
   Fold 3: 153577 train samples, 52443 val samples

fold_3...
  ✓ Accuracy: 0.1347
   Fold 4: 179769 train samples, 26251 val samples

fold_4...
  ✓ Accuracy: 0.3129


## Section 5: Statistical Evaluation Loop

In [6]:
# ---------------------------------------------------------
# SECTION 5: Fast Statistical Evaluation (No Data Leakage)
# ---------------------------------------------------------
results_all = []
alpha_corrected = CFG['alpha'] / len(CFG['models'])
n_folds = CFG['n_folds']

print(f"\n{'='*70}")
print(f"STATISTICAL EVALUATION: TRTR vs TSTR (Slide-Level Bootstrap)")
print(f"{'='*70}")
print(f"Bonferroni-corrected α = {alpha_corrected:.4f}\n")

for model_name, m_cfg in CFG['models'].items():
    print(f"\n{'-'*70}")
    print(f"MODEL: {model_name}")
    print(f"{'-'*70}")
    
    preds_A_list = []
    preds_B_list = []
    
    # 1. Generate strictly unseen predictions fold-by-fold
    for fold_idx in range(n_folds):
        X_val = X_val_list[fold_idx]
        
        try:
            # TRTR (Model A)
            path_A = find_model_path(CFG['model_a_dir'], m_cfg['prefix'], m_cfg['ext'], fold_idx)
            preds_A_list.append(load_and_predict(path_A, m_cfg['type'], X_val))
            
            # TSTR (Model B)
            path_B = find_model_path(CFG['model_b_dir'], m_cfg['prefix'], m_cfg['ext'], fold_idx)
            preds_B_list.append(load_and_predict(path_B, m_cfg['type'], X_val))
        except FileNotFoundError as e:
            print(f"  ✗ Missing model for {model_name} fold {fold_idx}")
            continue

    if len(preds_A_list) < n_folds or len(preds_B_list) < n_folds:
        print(f"  {model_name}: Incomplete predictions. Skipping.")
        continue

    # 2. Concatenate into unseen prediction arrays
    preds_A = np.concatenate(preds_A_list)
    preds_B = np.concatenate(preds_B_list)
    
    # --- SALVAGE SCRIPT: Map 6-class models down to Binary ---
    # Check if TRTR (Model A) needs mapping
    # if np.max(preds_A) > 1:
    #     print("  ! Mapping TRTR predictions from 6-class to binary...")
    #     preds_A = np.where(preds_A < 3, 0, 1)
        
    # # Check if TSTR/TSRTR (Model B) needs mapping
    # if np.max(preds_B) > 1:
    #     print("  ! Mapping TSTR/TSRTR predictions from 6-class to binary...")
    #     preds_B = np.where(preds_B < 3, 0, 1)
    # ---------------------------------------------------------
    
    # 3. Compute Baseline Metrics
    acc_A = accuracy_score(y_all, preds_A)
    acc_B = accuracy_score(y_all, preds_B)
    f1_A = f1_score(y_all, preds_A, average="macro", zero_division=0)
    f1_B = f1_score(y_all, preds_B, average="macro", zero_division=0)
    kappa_A = cohen_kappa_score(y_all, preds_A)
    kappa_B = cohen_kappa_score(y_all, preds_B)
    
    print(f"Aggregate Metrics (True Unseen):")
    print(f"  TRTR: Acc={acc_A:.4f}  F1={f1_A:.4f}  κ={kappa_A:.4f}")
    print(f"  TSTR: Acc={acc_B:.4f}  F1={f1_B:.4f}  κ={kappa_B:.4f}")
    print(f"  Δ: {acc_B - acc_A:+.4f}")
    
    # 4. Slide-level stratified bootstrap
    print(f"\nRunning slide-level bootstrap ({CFG['n_bootstrap']} iterations)...")
    boot_A, boot_B, boot_diffs = stratified_slide_bootstrap(
        preds_A, preds_B, y_all, slide_ids_all,
        n_boot=CFG['n_bootstrap'], seed=CFG['random_seed']
    )
    
    # 5. Bootstrap Statistics & Wilcoxon
    mean_diff = np.mean(boot_diffs)
    median_diff = np.median(boot_diffs)
    
    # Calculate CIs for Model A (TRTR)
    ci_A_lo = np.quantile(boot_A, 0.025)
    ci_A_hi = np.quantile(boot_A, 0.975)
    
    # Calculate CIs for Model B (TSTR)
    ci_B_lo = np.quantile(boot_B, 0.025)
    ci_B_hi = np.quantile(boot_B, 0.975)
    
    # Calculate CIs for the Difference
    ci_diff_lo = np.quantile(boot_diffs, 0.025)
    ci_diff_hi = np.quantile(boot_diffs, 0.975)

    # Calculate proportion of the bootstrap distribution that crosses 0
    prop_less = np.sum(boot_diffs <= 0) / len(boot_diffs)
    prop_greater = np.sum(boot_diffs >= 0) / len(boot_diffs)
    
    # Two-sided empirical p-value
    p_wilcoxon = 2.0 * min(prop_less, prop_greater) 
    
    # Effect Size Approximation (Cohen's d analogue for bootstrap)
    std_diff = np.std(boot_diffs)
    effect_r = abs(mean_diff) / std_diff if std_diff > 0 else 0.0
        
    sig_corrected = p_wilcoxon < alpha_corrected
    
    # 6. McNemar Test
    correct_A = (preds_A == y_all)
    correct_B = (preds_B == y_all)
    n_11 = np.sum(correct_A & correct_B)
    n_10 = np.sum(correct_A & ~correct_B)
    n_01 = np.sum(~correct_A & correct_B)
    n_00 = np.sum(~correct_A & ~correct_B)
    
    if n_10 + n_01 > 0:
        p_mcnemar = mcnemar([[n_11, n_10], [n_01, n_00]], exact=False, correction=True).pvalue
    else:
        p_mcnemar = 1.0
        
    print(f"\nBootstrap Distribution (Δ = TSTR - TRTR):")
    print(f"  Mean:   {mean_diff:+.4f}  [95% CI: {ci_diff_lo:+.4f}, {ci_diff_hi:+.4f}]")
    print(f"Wilcoxon Test: p={p_wilcoxon:.4f} (Significant: {'✓' if sig_corrected else '✗'})")
    
    # Store results
    results_all.append({
        "model": model_name, "acc_A": float(acc_A), "acc_B": float(acc_B),
        "acc_delta": float(acc_B - acc_A), "f1_A": float(f1_A), "f1_B": float(f1_B),
        "kappa_A": float(kappa_A), "kappa_B": float(kappa_B),
        "mean_diff": float(mean_diff), "median_diff": float(median_diff),
        "ci_A_lo": float(ci_A_lo), "ci_A_hi": float(ci_A_hi),
        "ci_B_lo": float(ci_B_lo), "ci_B_hi": float(ci_B_hi), 
        "ci_diff_lo": float(ci_diff_lo), "ci_diff_hi": float(ci_diff_hi),
        "p_wilcoxon": float(p_wilcoxon), "effect_r": float(effect_r),
        "sig_corrected": bool(sig_corrected), "p_mcnemar": float(p_mcnemar),
    })

# ============================================================
# CLASSIFICATION BY RECONSTRUCTION BOOTSTRAP (NEW)
# ============================================================
print(f"\n{'='*70}")
print(f"CLASSIFICATION BY RECONSTRUCTION: Bootstrap Confidence Intervals")
print(f"{'='*70}\n")

# Compute slide-level stratified bootstrap for CbR using global IDs
boot_acc_cbr = stratified_slide_bootstrap_cbr(
    cbr_preds, y_all, slide_ids_all,  # <--- Changed to global y_all and slide_ids_all
    n_boot=CFG['n_bootstrap'], seed=CFG['random_seed']
)

# Extract CbR bootstrap statistics
ci_cbr_lo = np.quantile(boot_acc_cbr, 0.025)
ci_cbr_hi = np.quantile(boot_acc_cbr, 0.975)
mean_acc_cbr = np.mean(boot_acc_cbr)

# --- THE FIX: Define missing baseline accuracy ---
cbr_acc_baseline = accuracy_score(y_all, cbr_preds)

print(f"\nCbR Bootstrap Results:")
print(f"  Baseline Accuracy: {cbr_acc_baseline:.4f}")
print(f"  Bootstrap Mean:    {mean_acc_cbr:.4f}")
print(f"  95% CI: [{ci_cbr_lo:.4f}, {ci_cbr_hi:.4f}]")

# Store CbR results
results_all.append({
    "model": "Classification_by_Reconstruction",
    "technique": "CVAE Signal Matching",
    "accuracy": float(cbr_acc_baseline),
    "mean_bootstrap": float(mean_acc_cbr),
    "ci_lo": float(ci_cbr_lo),
    "ci_hi": float(ci_cbr_hi),
    "n_iterations": CFG['n_bootstrap'],
    "seed": CFG['random_seed'],
    "n_slides": int(len(np.unique(slide_ids_all))), # <--- Changed
    "n_samples": int(len(cbr_preds)),
})


STATISTICAL EVALUATION: TRTR vs TSTR (Slide-Level Bootstrap)
Bonferroni-corrected α = 0.0125


----------------------------------------------------------------------
MODEL: Apollo
----------------------------------------------------------------------
Aggregate Metrics (True Unseen):
  TRTR: Acc=0.2537  F1=0.2090  κ=0.0948
  TSTR: Acc=0.3055  F1=0.2310  κ=0.1357
  Δ: +0.0518

Running slide-level bootstrap (1000 iterations)...
Running optimized slide-level bootstrap...

Bootstrap Distribution (Δ = TSTR - TRTR):
  Mean:   +0.0294  [95% CI: +0.0060, +0.0542]
Wilcoxon Test: p=0.0140 (Significant: ✗)

----------------------------------------------------------------------
MODEL: RF_Standard
----------------------------------------------------------------------
Aggregate Metrics (True Unseen):
  TRTR: Acc=0.2927  F1=0.2045  κ=0.1135
  TSTR: Acc=0.3211  F1=0.2098  κ=0.1366
  Δ: +0.0284

Running slide-level bootstrap (1000 iterations)...
Running optimized slide-level bootstrap...

Bootstrap Dis

## Section 6: Save & Summarize Results

In [ ]:
# Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
json_path = Path(CFG['output_dir']) / f"statistical_evaluation_{timestamp}.json"

with open(json_path, 'w') as f:
    json.dump(results_all, f, indent=2)

print(f"✓ Results saved: {json_path}")

# Summary table - include CbR in summary
summary_data = []

# Add TRTR/TSTR model results
for r in results_all:
    if r["model"] != "Classification_by_Reconstruction":
        summary_data.append({
            "Model": r["model"],
            "Acc(TRTR)": f"{r['acc_A']:.4f}",
            "Acc(TSTR)": f"{r['acc_B']:.4f}",
            "Δ": f"{r['acc_delta']:+.4f}",
            "p-val": f"{r['p_wilcoxon']:.4f}",
            "Sig": "✓" if r['sig_corrected'] else "",
            "Effect": f"{r['effect_r']:.3f}",
        })

# Add CbR result
for r in results_all:
    if r["model"] == "Classification_by_Reconstruction":
        summary_data.append({
            "Model": "CbR (CVAE)",
            "Acc(TRTR)": "",
            "Acc(TSTR)": f"{r['accuracy']:.4f}",
            "Δ": "",
            "p-val": "",
            "Sig": "",
            "Effect": "",
        })
        # Add CI note as separate row
        summary_data.append({
            "Model": "  ↳ 95% CI",
            "Acc(TRTR)": "",
            "Acc(TSTR)": f"[{r['ci_lo']:.4f}, {r['ci_hi']:.4f}]",
            "Δ": "",
            "p-val": "",
            "Sig": "",
            "Effect": "",
        })

summary_df = pd.DataFrame(summary_data)

print(f"\n{'='*70}")
print("SUMMARY")
print(f"{'='*70}")
print(summary_df.to_string(index=False))

csv_path = Path(CFG['output_dir']) / f"statistical_summary_{timestamp}.csv"
summary_df.to_csv(csv_path, index=False)
print(f"\n✓ Summary saved: {csv_path}")

# Also save a detailed CbR-specific report
cbr_detail = {
    "timestamp": timestamp,
    "technique": "CVAE Classification by Reconstruction",
    "baseline_accuracy": float(cbr_acc_baseline),
    "bootstrap_samples": CFG['n_bootstrap'],
    "bootstrap_mean_accuracy": float(mean_acc_cbr),
    "confidence_interval_95": {
        "lower": float(ci_cbr_lo),
        "upper": float(ci_cbr_hi),
    },
    "n_slides": int(len(np.unique(slide_ids_all))), # <--- Changed
    "n_total_samples": int(len(cbr_preds)),
    "n_folds": CFG['n_folds'],
    "random_seed": CFG['random_seed'],
    "methodology": "Slide-stratified bootstrap (cluster bootstrap at slide level)"
}

cbr_json_path = Path(CFG['output_dir']) / f"cbr_bootstrap_detail_{timestamp}.json"
with open(cbr_json_path, 'w') as f:
    json.dump(cbr_detail, f, indent=2)

print(f"✓ CbR details saved: {cbr_json_path}")

✓ Results saved: results\exp_104\statistical_eval\statistical_evaluation_20260513_115557.json

SUMMARY
      Model Acc(TRTR) Acc(TSTR)       Δ  p-val Sig Effect
     Apollo    0.2537    0.1009 -0.1528 1.0000      0.069
RF_Standard    0.2927    0.2027 -0.0900 1.0000      0.044
    XGBoost    0.3108    0.2391 -0.0716 1.0000      0.030
     1D_CNN    0.1772    0.0956 -0.0816 1.0000      0.062

✓ Summary saved: results\exp_104\statistical_eval\statistical_summary_20260513_115557.csv


## Section 7: Interpretation

In [ ]:
print(f"\n{'='*70}")
print("FINDINGS")
print(f"{'='*70}\n")

sig_models = [r for r in results_all if r.get('sig_corrected', False)]
improving = [r for r in results_all if r.get('acc_delta', 0) > 0]
degrading = [r for r in results_all if r.get('acc_delta', 0) < 0]

print(f"Statistically Significant Improvements (α = {alpha_corrected:.4f}):")
if sig_models:
    for r in sig_models:
        print(f"  • {r['model']:12s}: Δ = {r['acc_delta']:+.4f}, p = {r['p_wilcoxon']:.4f}")
else:
    print("  (None)")

print(f"\nImprovement Trend (not significant):")
for r in improving:
    if not r.get('sig_corrected', False):
        print(f"  • {r['model']:12s}: Δ = {r['acc_delta']:+.4f}, p = {r['p_wilcoxon']:.4f}")

if degrading:
    print(f"\nDegradation:")
    for r in degrading:
        print(f"  • {r['model']:12s}: Δ = {r['acc_delta']:+.4f}, p = {r['p_wilcoxon']:.4f}")

# CbR-specific findings
print(f"\n{'='*70}")
print("CLASSIFICATION BY RECONSTRUCTION (CbR) - CVAE Signal Matching")
print(f"{'='*70}")

for r in results_all:
    if r["model"] == "Classification_by_Reconstruction":
        print(f"\nCbR Accuracy (Baseline): {r['accuracy']:.4f}")
        print(f"Bootstrap Mean Accuracy: {r['mean_bootstrap']:.4f}")
        print(f"95% Confidence Interval: [{r['ci_lo']:.4f}, {r['ci_hi']:.4f}]")
        print(f"  Confidence Interval Width: {r['ci_hi'] - r['ci_lo']:.4f}")
        print(f"  slide cohort: {r['n_slides']} unique slides")
        print(f"  Total samples: {r['n_samples']} spectra")
        print(f"  Bootstrap iterations: {r['n_iterations']}")
        print(f"\nNote: CbR results are NOT compared statistically with TRTR/TSTR models.")
        print(f"      CIs provided for reference using same slide-stratified bootstrap methodology.")


FINDINGS

Statistically Significant Improvements (α = 0.0125):
  (None)

Improvement Trend (not significant):
  • 1D_CNN      : Δ = +0.0076, p = 0.1483

Degradation:
  • Apollo      : Δ = -0.1803, p = 1.0000
  • RF_Standard : Δ = -0.1133, p = 1.0000
  • XGBoost     : Δ = -0.0718, p = 1.0000
